
# Batch Region Division for **All Timestamps** in `cloud_echo_combined.csv`

This notebook loops over **every unique `time`** in your CSV and applies three methods:
1. **Fixed Grid + Uniqueness Check** — outputs regions per block with 1 optimal grid each (with uniqueness flag)
2. **Penalty-Based Clustering (KMeans)** — clusters cells and outputs the optimal grid per cluster
3. **Connected Severe Regions** — threshold-based connected components; outputs the optimal grid per region

For each timestamp, results are saved under `all_times_region_outputs/<ISO_TIME>/`:
- `method1_regions.csv`
- `method2_clusters.csv`
- `method3_regions.csv`

> Input CSV columns expected:
> `time, grid_id, mean_dbz, max_dbz, rain_flag, weather_level, weather_desc, is_heavy_cloud, has_tall_echo_top`


In [ ]:

from pathlib import Path
import os
from collections import deque
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------
# EDIT THESE PATHS/PARAMS
# ----------------------------
DATA_CSV = r"/Users/hoo/Desktop/UMICH/FAAFAST/radar_imgs_20231222_0000_0100/cloud_echo_combined.csv"

# Optional lookup mapping `grid_id -> row, col` (and optional lat,lon)
GRID_LOOKUP_CSV = None  # e.g., r"/path/to/grid_lookup.csv" with columns: grid_id,row,col[,lat,lon]

# If no lookup, set grid shape to infer row/col from grid_id (row-major).
NUM_ROWS = 5    # <-- EDIT to your grid layout
NUM_COLS = 5    # <-- EDIT to your grid layout

# Weather penalty weights
ALPHA = 1.0     # mean_dbz weight
BETA  = 0.2     # max_dbz weight
GAMMA = 2.0     # weather_level weight
DELTA = 5.0     # is_heavy_cloud add-on; 0.5*DELTA if has_tall_echo_top

# Threshold choice for "severe" (Method 3)
USE_PENALTY_THRESHOLD = True
PENALTY_THRESHOLD = 40.0
LEVEL_THRESHOLD   = 4

# KMeans clustering (Method 2)
KMEANS_CLUSTERS = 6

# Uniqueness tolerance
EPS = 1e-6

# Block size for Method 1 (coarser-than-cell regions if >1)
BLOCK_R = 1
BLOCK_C = 1

# Output base directory
BASE_OUT = Path.cwd() / "all_times_region_outputs"
BASE_OUT.mkdir(parents=True, exist_ok=True)
print("Outputs will be written under:", BASE_OUT)


In [ ]:

def safe_bool(x):
    if isinstance(x, bool):
        return x
    if isinstance(x, (int, float, np.integer, np.floating)):
        return bool(int(x))
    if isinstance(x, str):
        return x.strip().lower() in {"1","true","t","yes","y"}
    return False

def build_row_col_from_id(grid_id, n_rows, n_cols):
    r = grid_id // n_cols
    c = grid_id % n_cols
    return int(r), int(c)

def neighbors4(r, c, n_rows, n_cols):
    for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
        rr, cc = r+dr, c+dc
        if 0 <= rr < n_rows and 0 <= cc < n_cols:
            yield rr, cc

def connected_components(mask):
    n_rows, n_cols = mask.shape
    comp = -np.ones_like(mask, dtype=int)
    curr = 0
    from collections import deque
    for r in range(n_rows):
        for c in range(n_cols):
            if not mask[r, c] or comp[r, c] != -1:
                continue
            q = deque([(r,c)])
            comp[r, c] = curr
            while q:
                rr, cc = q.popleft()
                for nr, nc in neighbors4(rr, cc, n_rows, n_cols):
                    if mask[nr, nc] and comp[nr, nc] == -1:
                        comp[nr, nc] = curr
                        q.append((nr, nc))
            curr += 1
    return comp, curr


In [ ]:

# Load full CSV
df = pd.read_csv(DATA_CSV)

# Normalize boolean-like columns
for col in ["rain_flag","is_heavy_cloud","has_tall_echo_top"]:
    if col in df.columns:
        df[col] = df[col].apply(safe_bool)

times = df["time"].unique().tolist()
print("Total time slices:", len(times))
print("First 5 times:", times[:5])


In [ ]:

from sklearn.cluster import KMeans

def process_one_snapshot(snap: pd.DataFrame, out_dir: Path, num_rows: int, num_cols: int, has_latlon=False):
    # Penalty
    snap = snap.copy()
    snap["penalty"] = (
        ALPHA * snap["mean_dbz"].astype(float) +
        BETA  * snap["max_dbz"].astype(float) +
        GAMMA * snap["weather_level"].astype(float) +
        (DELTA * snap["is_heavy_cloud"].astype(float)) +
        (0.5 * DELTA * snap.get("has_tall_echo_top", pd.Series(False, index=snap.index)).astype(float))
    )

    grid_penalty = np.full((num_rows, num_cols), np.nan)
    for _, r in snap.iterrows():
        grid_penalty[int(r["row"]), int(r["col"])] = float(r["penalty"])

    # -------------------- Method 1 --------------------
    regions = []
    for r0 in range(0, num_rows, BLOCK_R):
        for c0 in range(0, num_cols, BLOCK_C):
            r1 = min(num_rows, r0 + BLOCK_R)
            c1 = min(num_cols, c0 + BLOCK_C)
            sub = []
            for rr in range(r0, r1):
                for cc in range(c0, c1):
                    val = grid_penalty[rr, cc]
                    if not np.isnan(val):
                        sub.append((rr, cc, val))
            if not sub:
                continue
            vals = np.array([v for (_,_,v) in sub], dtype=float)
            min_val = np.nanmin(vals)
            idxs = np.where(np.abs(vals - min_val) <= EPS)[0]

            if len(idxs) == 1:
                rr, cc, _ = sub[idxs[0]]
                unique = True
            else:
                cand = []
                for i in idxs:
                    rr, cc, _ = sub[i]
                    g = snap[(snap["row"]==rr) & (snap["col"]==cc)].iloc[0]
                    cand.append((rr, cc, float(g["mean_dbz"]), float(g["max_dbz"])))
                cand.sort(key=lambda t: (t[2], t[3], t[0], t[1]))
                rr, cc = cand[0][0], cand[0][1]
                unique = False

            members = [(rr2,cc2) for (rr2,cc2,_) in sub]
            member_grids = [int(snap[(snap["row"]==rr2)&(snap["col"]==cc2)]["grid_id"].iloc[0]) for (rr2,cc2) in members]
            opt_grid = int(snap[(snap["row"]==rr)&(snap["col"]==cc)]["grid_id"].iloc[0])

            regions.append({
                "block_row0": r0, "block_col0": c0,
                "block_row1": r1-1, "block_col1": c1-1,
                "member_grid_ids": member_grids,
                "optimal_grid_id": opt_grid,
                "optimal_row": int(rr), "optimal_col": int(cc),
                "penalty_min": float(grid_penalty[rr,cc]),
                "unique_min_before_tiebreak": unique
            })

    m1 = pd.DataFrame(regions)
    m1.to_csv(out_dir / "method1_regions.csv", index=False)

    # -------------------- Method 2 --------------------
    coords = snap[["row","col"]].values.astype(float)
    km = KMeans(n_clusters=KMEANS_CLUSTERS, random_state=42, n_init="auto")
    labels = km.fit_predict(coords)

    snap["cluster"] = labels
    cluster_opt_rows = []
    for cl in np.unique(labels):
        sub = snap[snap["cluster"]==cl].copy()
        idx = sub["penalty"].idxmin()
        row = sub.loc[idx]
        cluster_opt_rows.append({
            "cluster": int(cl),
            "optimal_grid_id": int(row["grid_id"]),
            "optimal_row": int(row["row"]),
            "optimal_col": int(row["col"]),
            "penalty_min": float(row["penalty"]),
            "member_grid_ids": list(map(int, sub["grid_id"].tolist()))
        })
    m2 = pd.DataFrame(cluster_opt_rows)
    m2.to_csv(out_dir / "method2_clusters.csv", index=False)

    # -------------------- Method 3 --------------------
    if USE_PENALTY_THRESHOLD:
        severe_mask = (grid_penalty >= PENALTY_THRESHOLD)
    else:
        level_grid = np.full((num_rows, num_cols), np.nan)
        for _, r in snap.iterrows():
            level_grid[int(r["row"]), int(r["col"])] = float(r["weather_level"])
        severe_mask = (np.nan_to_num(level_grid, nan=0.0) >= LEVEL_THRESHOLD)

    comp_id, n_comp = connected_components(severe_mask)

    rows = []
    for rid in range(n_comp):
        idx = np.argwhere(comp_id == rid)
        member_ids = []
        cand = []
        for (rr, cc) in idx:
            g = snap[(snap["row"]==rr) & (snap["col"]==cc)].iloc[0]
            member_ids.append(int(g["grid_id"]))
            cand.append((rr, cc, float(g["penalty"]), float(g["mean_dbz"]), float(g["max_dbz"])))
        cand = sorted(cand, key=lambda t: (t[2], t[3], t[4], t[0], t[1]))
        rr, cc, pmin, _, _ = cand[0]
        opt_grid = int(snap[(snap["row"]==rr)&(snap["col"]==cc)]["grid_id"].iloc[0])
        rows.append({
            "region_id": rid,
            "optimal_grid_id": opt_grid,
            "optimal_row": int(rr), "optimal_col": int(cc),
            "penalty_min": float(pmin),
            "member_grid_ids": member_ids
        })

    m3 = pd.DataFrame(rows)
    m3.to_csv(out_dir / "method3_regions.csv", index=False)

    # Simple visualization saved
    fig, ax = plt.subplots(figsize=(6,5))
    im = ax.imshow(grid_penalty, origin='lower')
    ys, xs = np.where(severe_mask)
    ax.scatter(xs, ys, s=12, marker='s', facecolors='none')
    ax.set_title("Penalty heatmap with severe cells")
    ax.set_xlabel("col")
    ax.set_ylabel("row")
    fig.tight_layout()
    fig.savefig(out_dir / "quick_visualization.png")
    plt.close(fig)


In [ ]:

# MAIN LOOP over all times
df_all = df.copy()

# Build row/col mapping once (using lookup or inference)
if GRID_LOOKUP_CSV and os.path.exists(GRID_LOOKUP_CSV):
    lut = pd.read_csv(GRID_LOOKUP_CSV)
    need = {"grid_id","row","col"}
    if not need.issubset(set(lut.columns)):
        raise ValueError("grid_lookup must have columns: grid_id,row,col (optional lat,lon)")
    df_all = df_all.merge(lut, on="grid_id", how="left")
    has_latlon = {"lat","lon"}.issubset(df_all.columns)
else:
    rc = df_all["grid_id"].astype(int).apply(lambda gid: build_row_col_from_id(gid, NUM_ROWS, NUM_COLS))
    df_all["row"] = rc.apply(lambda t: t[0])
    df_all["col"] = rc.apply(lambda t: t[1])
    has_latlon = False

n_rows = int(df_all["row"].max()) + 1
n_cols = int(df_all["col"].max()) + 1
print(f"Inferred grid: {n_rows} x {n_cols}")

times_sorted = sorted(df_all["time"].unique().tolist())
for t in times_sorted:
    snap = df_all[df_all["time"] == t].copy().reset_index(drop=True)
    # Create ISO-like safe dir name
    safe = str(t).replace(":", "-").replace(" ", "_")
    out_dir = (BASE_OUT / safe)
    out_dir.mkdir(parents=True, exist_ok=True)
    print("Processing", t, "->", out_dir)
    process_one_snapshot(snap, out_dir, n_rows, n_cols, has_latlon=has_latlon)

print("Done. See outputs under:", BASE_OUT)
